# G-Research Crypto Forecasting - Exploratory Analysis

Four questions, each of which changes a modelling decision:

1. How much of the minute grid is actually missing?
2. How strongly do the coins move together? (This is why the target is residualised.)
3. What does residualisation do to the target?
4. Is there anything predictable, and does it need information from *other* coins?

> Reads the synthetic data by default. Point `DATA_DIR` at the real files to switch.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from cryptoforecast.data import load_grid
from cryptoforecast.constants import ASSET_NAMES
from cryptoforecast.target import forward_fill, forward_log_return, market_return
from cryptoforecast.features import backward_log_return

plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
# Plot labels are plain ASCII: many environments lack extra fonts

DATA_DIR = '../data'
grid = load_grid(DATA_DIR)
names = [ASSET_NAMES[a] for a in grid.asset_ids]

## 1. Missing minutes

The raw file only has a row when a coin traded. On that frame, "16 rows
ahead" is not "16 minutes ahead" - which is why everything here runs on a
complete minute grid.

In [ ]:
cov = grid.coverage()
order = sorted(cov, key=cov.get)
fig, ax = plt.subplots()
ax.barh([ASSET_NAMES[a] for a in order], [1 - cov[a] for a in order], color='indianred')
ax.set_xlabel('fraction of minutes with no trade'); ax.set_title('Missing minutes by coin')
plt.tight_layout(); plt.show()

for a in order[:3]:
    print(f'{ASSET_NAMES[a]:<18} trades in {cov[a]:.1%} of minutes')

## 2. How much do the coins move together?

In [ ]:
close = forward_fill(grid.fields['Close'], limit=30)
r15 = backward_log_return(close, 15)[::15]          # non-overlapping 15-minute returns
ok = np.isfinite(r15).all(axis=1)
corr = np.corrcoef(r15[ok].T)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=7)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
plt.colorbar(im); ax.set_title('Correlation of 15-minute returns')
plt.tight_layout(); plt.show()

off = corr[~np.eye(len(corr), dtype=bool)]
print(f'mean pairwise correlation: {off.mean():.3f}')

A mean pairwise correlation this high means most of any coin's move is
simply "the market". Predicting raw returns would mostly mean predicting the
market. Residualising strips that common part out, so the target asks a
harder and more useful question: *which coin will beat the others?*

## 3. What residualisation does

In [ ]:
R = forward_log_return(close)
M = market_return(R, grid.weight_vector())
T = grid.target

j = grid.asset_ids.index(4)     # Dogecoin
ok = np.isfinite(R[:, j]) & np.isfinite(T[:, j]) & np.isfinite(M)
print(f'corr(raw 15m return, market)      : {np.corrcoef(R[ok, j], M[ok])[0,1]:+.3f}')
print(f'corr(residualised target, market) : {np.corrcoef(T[ok, j], M[ok])[0,1]:+.3f}')
print(f'std raw return / std target       : {np.std(R[ok, j]) / np.std(T[ok, j]):.2f}')

The residual is nearly uncorrelated with the market - which is the point -
and much less volatile than the raw return, because most of the variance
was the common factor.

## 4. Is anything predictable, and does it need other coins?

In [ ]:
# Own information: does a coin's recent drift relative to the market predict
# its next residual move? (mean reversion)
rel = backward_log_return(close, 60) - market_return(backward_log_return(close, 60),
                                                    grid.weight_vector())[:, None]
# Cross information: does Bitcoin's last few minutes predict a smaller coin?
btc = grid.asset_ids.index(1)
lead = backward_log_return(close[:, [btc]], 3)[:, 0]

print(f'{"coin":<18}{"own: rel_ret_60":>18}{"cross: BTC 3m":>16}')
for a in grid.asset_ids:
    j = grid.asset_ids.index(a)
    ok1 = np.isfinite(rel[:, j]) & np.isfinite(T[:, j])
    ok2 = np.isfinite(lead) & np.isfinite(T[:, j])
    c1 = np.corrcoef(rel[ok1, j], T[ok1, j])[0, 1]
    c2 = np.corrcoef(lead[ok2], T[ok2, j])[0, 1]
    print(f'{ASSET_NAMES[a]:<18}{c1:>+18.4f}{c2:>+16.4f}')

Two different sources of signal, with different implications:

* **Own-history mean reversion** shows up for every coin. Any model can use it.
* **Lead-lag from Bitcoin** shows up only for the coins that react to the
  market with a delay, and it is invisible to a model that looks at one coin
  at a time. This is the structural argument for the cross-asset network.

## What the data implies for the model

| Observation | Decision |
|---|---|
| Coins skip minutes | Put everything on a complete minute grid first |
| Target starts at t+1, ends at t+16 | Embargo of at least 16 minutes between train and validation |
| Coins are highly correlated | The target is relative; model relative structure |
| Every coin's target depends on all coins' futures | No market statistic built from forward prices may be a feature |
| Lead-lag exists for slow coins | Give models access to the other coins' recent moves |
| Metric is a per-coin correlation | Prediction scale is irrelevant; standardise targets per coin |